Steps
1. Chunk
2. Embed each chunk with 

In [1]:
from dotenv import load_dotenv
import os
load_dotenv(override=True) 

True

# Prep data - convert PDF to markdown

In [2]:
# unlock the PDF file
import pikepdf
from markitdown import MarkItDown

with pikepdf.open("data/locked_JJ_10K.pdf") as pdf:
    pdf.save("data/unlocked_JJ_10K.pdf")
# convert the PDF to markdown
md = MarkItDown()
results = md.convert("data/unlocked_JJ_10K.pdf")
# save the markdown to a file
with open("data/JJ_10K.md", "w", encoding='utf-8') as f:
    f.write(results.text_content)

# note: Claude parsed markdown better than Markitdown,
# will use claude output for now. need to clean up pdf parsing and markdown conversion in the future.

c:\Users\Josiah\Git_Repos\EasyRag\.venv\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


KeyboardInterrupt: 

# Vectorize Document

In [12]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import openai
from util import Embedder
import os

In [3]:
class MockUploadedFile:
    def __init__(self, path):
        self.name = path.split("/")[-1]   # or os.path.basename(path)
        self.type = self._get_mime()
    
    def _get_mime(self):
        ext = self.name.split(".")[-1].lower()
        mime_map = {
            "txt":  "text/plain",
            "md":   "text/markdown",
            "pdf":  "application/pdf",
            "docx": "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
        }
        return mime_map.get(ext, "application/octet-stream")
    
    def read(self):
        with open(self._path, "rb") as f:
            return f.read()
    
    def __init__(self, path):
        self._path = path
        self.name = path.split("\\")[-1].split("/")[-1]
        self.type = self._get_mime()

# Usage in notebook
mock_file = MockUploadedFile("./data/example.txt")

In [4]:
Embedder = Embedder()
print('current collections:', Embedder.collection_names)

Initializing Embedder with model text-embedding-3-small
current collections: []


In [ ]:
Embedder.embed_file(mock_file)
print(Embedder.collection_names)

Attempting to embed file: example.txt of type text/plain
Embedding file: example.txt of type text/plain


(True, 'File example.txt embedded successfully.')

In [10]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
import os

# 1. Create embeddings (LangChain style)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY")
)

# 2. Connect to existing Chroma DB
vectorstore = Chroma(
    persist_directory="./chroma_db",   # same path as before
    embedding_function=Embedder.embedding_model,
    collection_name="example-txt"  # same collection name as before
)

print("Available collections:", vectorstore._client.list_collections())

# 3. Query
results = vectorstore.similarity_search(
    "What is the purpose of this document?",
    k=3
)



Available collections: [Collection(name=langchain), Collection(name=example-txt)]


In [11]:
for r in results:
    print(r.page_content)
    print(20*"---")
    print(20*"---")
    print()

- Janssen filed litigation in July 2023 challenging the IRA's constitutionality under the First and Fifth Amendments; appeal filed to the Third Circuit in April 2024.
- EU regulations: NIS2, EHDS, Data Act, Cyber Resilience Act, AI Act — increasing privacy and cybersecurity compliance requirements.
- China data security and personal information protection regulations.
- 340B Drug Pricing Program requirements.

#### Employees and Human Capital Management

| Metric | 2024 | 2023 |
------------------------------------------------------------
------------------------------------------------------------

- U.S. FDA regulation of pharmaceutical products and medical technology (product safety, efficacy, manufacturing, advertising, labeling, safety reporting)
- Inflation Reduction Act (IRA, 2022): CMS authorized to negotiate Medicare prices for high-spend drugs starting in 2026 (Part D) and 2028 (Part B). XARELTO, STELARA, and IMBRUVICA are on the first Selected Drug list.
--------------------

In [4]:
import chromadb

# Connect to your ChromaDB (adjust path if using persistent storage)
client = chromadb.PersistentClient(path="./chroma_db" )  # or chromadb.Client() for in-memory

# List all collections
print("Collections:", client.list_collections())

# Get a specific collection
try:
    collection = client.get_collection(name="example-txt")
except Exception as e:
    print(f"Error retrieving collection: {e}")
    collection = None

# Check basic info
if collection:
    print("Count:", collection.count())

# Peek at a few items
if collection:
    print(collection.peek(limit=5))

Collections: []
Error retrieving collection: Collection [example-txt] does not exist


In [1]:
import util
import util.agent
import util.embedder

embedder = util.embedder.Embedder()
agent = util.agent.build_agent()



In [2]:
agent.SkillRegistry.embedder.embed_file("data/example.txt")

(False, 'Duplicate file: example.txt already embedded.')

In [5]:
agent_response = agent.run("find me a funny joke from the internet and where to fine it" )

In [6]:
agent_response

'Here\'s a funny joke I found from the internet, along with a couple of bonus ones! 😄\n\n---\n\n🎉 **Joke #1:**\n\n"I\'m reading a book on anti-gravity — it\'s impossible to put down.\n\nI told my suitcase we weren\'t going on vacation. Now it\'s dealing with emotional baggage."\n\n📍 **Where to find it:** [punscorner.com/new-and-funny-jokes](https://punscorner.com/new-and-funny-jokes/)\n\n---\n\n🎉 **Bonus Jokes:**\n\n**Why don\'t scientists trust atoms?** Because they make up everything.\n\n**Why don\'t eggs tell jokes?** They\'d crack each other up.\n\n📍 **Where to find more:** [funnnypuns.com/funny-jokes-2](https://funnnypuns.com/funny-jokes-2/) and [jokedrops.com/2025-short-funny-jokes](https://jokedrops.com/2025-short-funny-jokes/)\n\n---\n\nHope these gave you a good laugh! 😂 Let me know if you want more jokes in a specific style (dad jokes, puns, riddles, etc.)!'

In [3]:
agent_response = agent.run("What is the purpose of the document in the example-txt collection?")

In [4]:
agent_response

"The document in the **example-txt** collection is **Johnson & Johnson's Annual Report on Form 10-K for Fiscal Year 2024**. Here is a summary of its purpose:\n\n- It is formally titled **Johnson & Johnson — Form 10-K (FY2024)**, filed on **February 13, 2025**, for the fiscal year ended **December 29, 2024**, under the ticker **JNJ** on the NYSE, sourced from SEC EDGAR.\n\n- A **Form 10-K** is a comprehensive annual report required by the U.S. Securities and Exchange Commission (SEC). It is intended to give shareholders, investors, and the public a detailed overview of a public company's financial performance and business operations.\n\n- The document covers Johnson & Johnson's two major business segments — **Innovative Medicine** and **MedTech** — and describes how the company is organized, its strategy, and accountability structures.\n\n- It also addresses key regulatory and legal matters, such as litigation related to the Inflation Reduction Act's constitutionality and compliance wit